# Estudo de Estacionaridade das Séries do PIB-Nowcast

Objetivo: testar diferentes especificações de transformação para cada variável do modelo e identificar a mais parcimoniosa que atinge estacionaridade.

> **Disclaimer:** As transformações abaixo são aplicadas sobre as séries **já com ajuste sazonal**. Variáveis que possuem ajuste sazonal de origem (da fonte) são usadas diretamente; as demais passam pelo ajuste sazonal via X-13ARIMA-SEATS conforme implementado na função `seas_adj`.

**Transformações candidatas:**
- Log, Box-Cox, Yeo-Johnson
- 1ª diferença, diferença sazonal (12 para mensal, 4 para trimestral)
- Combinações cumulativas

**Critério:** maioria dos testes (ADF, KPSS, PP, DFGLS) indica estacionaridade. Em caso de empate entre especificações, a mais parcimoniosa vence.

> **Nota:** A variável `pib` usa variação percentual YoY (4 trimestres), **não** dlog4.

In [1]:
# %% Setup
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
for pr in [Path.cwd(), Path.cwd().parent]:
    if (pr / 'pib_nowcast').is_dir():
        sys.path.insert(0, str(pr))
        break

import numpy as np
import pandas as pd
import datetime as dt

from pib_nowcast.config import SERIES_SPEC, LAST_DATA, DATA_DIR
from pib_nowcast.utils.transformations import (
    stationarity_tests,
    is_stationary,

    seas_adj,
    seas_adj_parallel,
    seas_adj_stl, 
    seas_adj_stl_parallel,
    
    PIPELINE_REGISTRY,
    PIPELINE_NAME_TO_ID,
    MONTHLY_PIPELINE_IDS,
    QUARTERLY_PIPELINE_IDS,
    apply_transform_pipeline,
)
from pib_nowcast.utils.get_data import get_data

In [2]:
start = dt.datetime.now()

In [3]:
# non_sa_vars = specs_df.query("seas_adj == 0")["variable"].to_list()
# base_df_sa = full_data.copy(deep=True)

# freq_map = {
# 'Quarterly': 4,
# 'Monthly': 12
#         }


# jobs = {}
# for col in non_sa_vars:
#     if col not in base_df_sa.columns:
#         logger.warning(f"Coluna '{col}' não encontrada no DataFrame, pulando.")
#         continue

#     print(col)

#     series = base_df_sa[[col]].dropna()
#     freq_str = specs_df.loc[specs_df['variable'] == col, 'frequency'].iloc[0]
    
#     display(freq_str)

#     period = freq_map.get(freq_str)
#     display(period)

#     break

    
    


In [4]:
# %% Carregar dados (já com ajuste sazonal)
specs_df = pd.read_csv(SERIES_SPEC, sep=';')
start_date = '1996-01-01'

try:
    # Coleta de arquivo persistido
    full_data = pd.read_excel(LAST_DATA, sheet_name='full_dataset', index_col='Date')

except:
    # Coleta via API
    full_data = get_data(specs_df, start_date)

# Ajuste sazonal
# full_data = seas_adj(full_data, specs_df)
full_data = seas_adj_stl_parallel(full_data, specs_df)

# Frequência por variável
freq_map = specs_df.set_index('variable')['frequency'].to_dict()

print(f"Variáveis: {list(full_data.columns)}")
print(f"Período: {full_data.index.min()} a {full_data.index.max()}")
print(f"Shape: {full_data.shape}")
display(full_data.head())
display(full_data.tail())

Variáveis: ['ind_extrativa', 'ind_transformacao', 'ind_bens_capital', 'ind_bens_intermediarios', 'ind_bens_consumo', 'ind_construcao', 'cons_energ_comercial', 'cons_energ_residencial', 'cons_energ_industrial', 'cons_energ_outros', 'pmc_ampliada', 'ibcbr_agro', 'ibcbr_ind', 'ibcbr_servicos', 'icc_fecomercio', 'icea_fecomercio', 'ief_fecomercio', 'ics_fgv', 'isas_fgv', 'ies_fgv', 'caged_estoque', 'pnadc_salario_real_efetivo', 'pnadc_salario_real_com_carteira', 'pnadc_salario_real_sem_carteira', 'pnadc_salario_real_setor_privado', 'pnadc_salario_real_setor_publico', 'pnadc_salario_real_conta_propria', 'pnadc_salario_real_empregadores', 'pnadc_massa_salarial_efetiva', 'pnadc_massa_salarial_habitual', 'renda_disponivel_familias_restrita', 'tx_desemprego', 'ipca', 'ipca_alimentacao', 'ipca_habitacao', 'ipca_artigos_resid', 'ipca_vestuario', 'ipca_transportes', 'ipca_saude', 'ipca_despesas_pess', 'ipca_educacao', 'ipca_comunicacao', 'ipca_alim_domicilio', 'ipca_industriais', 'ipca_servicos', 

,ind_extrativa,ind_transformacao,ind_bens_capital,ind_bens_intermediarios,ind_bens_consumo,ind_construcao,cons_energ_comercial,cons_energ_residencial,cons_energ_industrial,cons_energ_outros,...,cni_vendas_industriais,cni_horas_trabalhadsa,cni_salario_industria,cni_massa_salarial,fenabrave_automoveis,fenabrave_comerciais_leves,fenabrave_caminhoes,fenabrave_onibus,fgv_nuci,pib
Date,,,,,,,,,,,,,,,,,,,,,
1996-01-01,NaN,NaN,NaN,NaN,NaN,NaN,2669.252341,5352.287733,9468.615850,2669.252341,...,NaN,NaN,NaN,NaN,104317.078340,19799.397055,3523.768436,1404.445098,NaN,NaN
1996-02-01,NaN,NaN,NaN,NaN,NaN,NaN,2853.114651,5626.543811,9566.559408,2853.114651,...,NaN,NaN,NaN,NaN,117067.157717,23269.548866,3840.335996,1340.005544,NaN,NaN
1996-03-01,NaN,NaN,NaN,NaN,NaN,NaN,2887.607095,5746.640641,9718.307832,2887.607095,...,NaN,NaN,NaN,NaN,112245.151895,23497.119953,3661.551995,1342.383177,NaN,96.84
1996-04-01,NaN,NaN,NaN,NaN,NaN,NaN,2905.501797,5767.617860,9518.251763,2905.501797,...,NaN,NaN,NaN,NaN,96226.323303,19868.307415,3377.949030,1372.317846,NaN,NaN
1996-05-01,NaN,NaN,NaN,NaN,NaN,NaN,2904.763141,5725.958721,9518.689289,2904.763141,...,NaN,NaN,NaN,NaN,110554.555936,21763.933465,3859.578292,1377.253035,NaN,NaN


,ind_extrativa,ind_transformacao,ind_bens_capital,ind_bens_intermediarios,ind_bens_consumo,ind_construcao,cons_energ_comercial,cons_energ_residencial,cons_energ_industrial,cons_energ_outros,...,cni_vendas_industriais,cni_horas_trabalhadsa,cni_salario_industria,cni_massa_salarial,fenabrave_automoveis,fenabrave_comerciais_leves,fenabrave_caminhoes,fenabrave_onibus,fgv_nuci,pib
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-01,118.1,103.4,91.7,106.1,107.0,99.4,8704.638218,15092.569377,16532.465238,8704.638218,...,110.7,94.0,111.6,111.6,176828.748568,46516.954212,8230.154463,1936.561448,81.6,NaN
2026-03-01,118.6,103.6,92.3,106.7,107.5,100.9,8686.237099,14963.986868,16463.327688,8686.237099,...,115.0,94.7,107.8,107.8,227912.062430,57313.392537,8786.012409,2348.182482,82.4,194.53
2026-04-01,122.3,103.9,92.4,108.3,106.8,99.2,9093.055675,15878.109552,16724.386191,9093.055675,...,115.6,93.5,113.2,113.2,193981.614044,50587.384363,8828.897087,2258.512039,83.2,NaN
2026-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,209143.353870,49162.382123,8744.321306,2081.736497,82.8,NaN
2026-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Registro de Pipelines de Transformação

Cada pipeline possui um **ID numérico** fixo definido em `PIPELINE_REGISTRY`. A tabela abaixo lista todos os pipelines disponíveis.

In [5]:
# %% Tabela de referência dos pipelines
reg_rows = []
for pid, (name, steps, needs_pos) in sorted(PIPELINE_REGISTRY.items()):
    grupo = 'Trimestral' if pid >= 13 else 'Mensal'
    if pid in MONTHLY_PIPELINE_IDS and pid in QUARTERLY_PIPELINE_IDS:
        grupo = 'Ambos'
    reg_rows.append({
        'pipeline_id': pid,
        'pipeline': name,
        'n_steps': len(steps),
        'requer_positivo': '✓' if needs_pos else '',
        'frequência': grupo,
    })

pd.DataFrame(reg_rows)

,pipeline_id,pipeline,n_steps,requer_positivo,frequência
0,0,nível,0,,Ambos
1,1,diff,1,,Ambos
2,2,log,1,✓,Ambos
3,3,log→diff,3,✓,Ambos
4,4,boxcox,1,✓,Mensal
5,5,boxcox→diff,2,✓,Mensal
6,6,yeojohnson,1,,Mensal
7,7,yeojohnson→diff,2,,Mensal
8,8,sdiff12,1,,Mensal
9,9,sdiff12→diff,2,,Mensal


## Teste de todas as combinações variável × pipeline

In [6]:
# %% Loop por variável
all_results = []

for col in full_data.columns:
    raw = full_data[col].dropna()
    freq = freq_map.get(col, 'Monthly')
    pipe_ids = QUARTERLY_PIPELINE_IDS if freq == 'Quaterly' else MONTHLY_PIPELINE_IDS

    for pid in pipe_ids:
        pipe_name, steps, _ = PIPELINE_REGISTRY[pid]
        transformed = apply_transform_pipeline(raw, pid)

        if transformed is None:
            all_results.append({
                'variable': col,
                'frequency': freq,
                'pipeline_id': pid,
                'pipeline': pipe_name,
                'n_steps': len(steps),
                'n_obs': 0,
                'ADF': None,
                'KPSS': None,
                'Phillips-Perron': None,
                'DFGLS': None,
                'n_stationary': 0,
                'is_stationary': False,
                'applicable': False,
            })
            continue

        try:
            tests_df = stationarity_tests(transformed)
            test_results = tests_df.set_index('test')['is_stationary'].to_dict()
            # Contar apenas testes que rodaram (não None)
            valid_results = {k: v for k, v in test_results.items() if v is not None}
            n_stat = sum(valid_results.values())
            n_valid = len(valid_results)
            is_stat = n_stat > n_valid / 2 if n_valid > 0 else False
        except Exception as e:
            print(f'  ⚠ {col}/{pipe_name}: erro nos testes ({type(e).__name__}), pulando.')
            all_results.append({
                'variable': col,
                'frequency': freq,
                'pipeline_id': pid,
                'pipeline': pipe_name,
                'n_steps': len(steps),
                'n_obs': len(transformed),
                'ADF': None,
                'KPSS': None,
                'Phillips-Perron': None,
                'DFGLS': None,
                'n_stationary': 0,
                'is_stationary': False,
                'applicable': False,
            })
            continue

        all_results.append({
            'variable': col,
            'frequency': freq,
            'pipeline_id': pid,
            'pipeline': pipe_name,
            'n_steps': len(steps),
            'n_obs': len(transformed),
            'ADF': test_results.get('ADF'),
            'KPSS': test_results.get('KPSS'),
            'Phillips-Perron': test_results.get('Phillips-Perron'),
            'DFGLS': test_results.get('DFGLS'),
            'n_stationary': n_stat,
            'is_stationary': is_stat,
            'applicable': True,
        })

    print(f'✓ {col}')

results_df = pd.DataFrame(all_results)
print(f'\nTotal de combinações testadas: {len(results_df)}')
print(f'Inaplicáveis: {(~results_df["applicable"]).sum()}')

✓ ind_extrativa
✓ ind_transformacao
✓ ind_bens_capital
✓ ind_bens_intermediarios
✓ ind_bens_consumo
✓ ind_construcao
✓ cons_energ_comercial
✓ cons_energ_residencial
✓ cons_energ_industrial
✓ cons_energ_outros
✓ pmc_ampliada
✓ ibcbr_agro
✓ ibcbr_ind
✓ ibcbr_servicos
✓ icc_fecomercio
✓ icea_fecomercio
✓ ief_fecomercio
✓ ics_fgv
✓ isas_fgv
✓ ies_fgv
✓ caged_estoque
✓ pnadc_salario_real_efetivo
✓ pnadc_salario_real_com_carteira
✓ pnadc_salario_real_sem_carteira
✓ pnadc_salario_real_setor_privado
✓ pnadc_salario_real_setor_publico
✓ pnadc_salario_real_conta_propria
✓ pnadc_salario_real_empregadores
✓ pnadc_massa_salarial_efetiva
✓ pnadc_massa_salarial_habitual
✓ renda_disponivel_familias_restrita
✓ tx_desemprego
✓ ipca
✓ ipca_alimentacao
✓ ipca_habitacao
✓ ipca_artigos_resid
✓ ipca_vestuario
✓ ipca_transportes
✓ ipca_saude
✓ ipca_despesas_pess
✓ ipca_educacao
✓ ipca_comunicacao
✓ ipca_alim_domicilio
✓ ipca_industriais
✓ ipca_servicos
✓ ipca_administrados
✓ ipca_livres
✓ ipca_nao_duraveis
✓ 

## Tabela Resumo

Para cada variável e pipeline, mostra se cada teste indicou estacionaridade (✓/✗) e o consenso geral.

In [7]:
# %% Tabela resumo formatada

def fmt_bool(v):
    if v is None:
        return '—'
    return '✓' if v else '✗'

display_df = results_df.copy()
for test_col in ['ADF', 'KPSS', 'Phillips-Perron', 'DFGLS', 'is_stationary']:
    display_df[test_col] = display_df[test_col].map(fmt_bool)

display_df = display_df.rename(columns={
    'variable': 'Variável',
    'pipeline_id': 'ID',
    'pipeline': 'Pipeline',
    'n_steps': 'Etapas',
    'n_obs': 'N obs',
    'n_stationary': 'Votos',
    'is_stationary': 'Estacionária?',
    'applicable': 'Aplicável',
})

display_df[['Variável', 'ID', 'Pipeline', 'Etapas', 'N obs', 'ADF', 'KPSS', 'Phillips-Perron', 'DFGLS', 'Votos', 'Estacionária?']]

,Variável,ID,Pipeline,Etapas,N obs,ADF,KPSS,Phillips-Perron,DFGLS,Votos,Estacionária?
0,ind_extrativa,0,nível,0,292,✗,✗,✗,✗,0,✗
1,ind_extrativa,1,diff,1,291,✓,✓,✓,✓,4,✓
2,ind_extrativa,2,log,1,292,✓,✗,✗,✗,1,✗
3,ind_extrativa,3,log→diff,3,291,✓,✓,✓,✓,4,✓
4,ind_extrativa,4,boxcox,1,292,✗,✗,✗,✗,0,✗
...,...,...,...,...,...,...,...,...,...,...,...
1489,pib,15,yoy_pct,1,117,✗,✓,✓,✓,3,✓
1490,pib,16,sdiff4,1,117,✗,✓,✓,✓,3,✓
1491,pib,17,sdiff4→diff,2,116,✓,✓,✓,✓,4,✓
1492,pib,18,log→sdiff4,3,117,✗,✓,✓,✓,3,✓


## Recomendação

Para cada variável, selecionar a transformação **mais parcimoniosa** (menor nº de etapas) dentre as estacionárias.

Em caso de empate de parcimônia, preferências:
1. `log` sobre `boxcox`/`yeojohnson` (mais interpretável)
2. Mais votos de estacionaridade
3. Mais observações preservadas

In [8]:
# %% Selecionar melhor pipeline por variável

# Ordem de preferência para desempate (menor = melhor)
PIPELINE_PREFERENCE = {
    # Mensais
    'nível': 0,
    'diff': 1,
    'log': 2,
    'log→diff': 3,
    'sdiff12': 4,
    'log→sdiff12': 5,
    'yeojohnson': 6,
    'yeojohnson→diff': 7,
    'boxcox': 8,
    'boxcox→diff': 9,
    'sdiff12→diff': 10,
    'log→sdiff12→diff': 11,
    'boxcox→sdiff12→diff': 12,
    # Trimestrais
    'yoy_pct': 1,
    'sdiff4': 4,
    'sdiff4→diff': 10,
    'log→sdiff4': 5,
    'log→sdiff4→diff': 11,
}

stationary = results_df[results_df['is_stationary'] & results_df['applicable']].copy()
stationary['preference'] = stationary['pipeline'].map(PIPELINE_PREFERENCE).fillna(99)

# Ordenar: menos etapas → mais preferido → mais votos → mais obs
stationary = stationary.sort_values(
    ['n_steps', 'preference', 'n_stationary', 'n_obs'],
    ascending=[True, True, False, False],
)

# Melhor por variável
best = stationary.groupby('variable').first().reset_index()
best = best[['variable', 'frequency', 'pipeline_id', 'pipeline', 'n_steps', 'n_obs', 'n_stationary']]
best.columns = ['Variável', 'Frequência', 'Pipeline ID', 'Pipeline Recomendado', 'Etapas', 'N obs', 'Votos']

print('═' * 80)
print('  RECOMENDAÇÃO DE TRANSFORMAÇÃO POR VARIÁVEL')
print('═' * 80)
best

════════════════════════════════════════════════════════════════════════════════
  RECOMENDAÇÃO DE TRANSFORMAÇÃO POR VARIÁVEL
════════════════════════════════════════════════════════════════════════════════


,Variável,Frequência,Pipeline ID,Pipeline Recomendado,Etapas,N obs,Votos
0,abras,Monthly,1,diff,1,303,3
1,caged_estoque,Monthly,1,diff,1,363,4
2,cni_horas_trabalhadsa,Monthly,1,diff,1,279,4
3,cni_massa_salarial,Monthly,1,diff,1,243,3
4,cni_nuci_transformacao,Monthly,1,diff,1,279,4
...,...,...,...,...,...,...,...
102,renda_disponivel_familias_restrita,Monthly,1,diff,1,277,4
103,tx_desemprego,Monthly,1,diff,1,170,3
104,vendas_autoveiculos_externo,Monthly,1,diff,1,254,4
105,vendas_autoveiculos_interno,Monthly,0,nível,0,255,3


In [9]:
# %% Variáveis sem nenhuma transformação estacionária
all_vars = set(full_data.columns)
stationary_vars = set(stationary['variable'].unique())
problematic = all_vars - stationary_vars

if problematic:
    print('⚠️  Variáveis sem NENHUMA transformação estacionária:')
    for v in sorted(problematic):
        print(f'   - {v}')
else:
    print('✅ Todas as variáveis possuem pelo menos uma transformação estacionária.')

✅ Todas as variáveis possuem pelo menos uma transformação estacionária.


In [10]:
# %% Resumo por variável: todas as opções estacionárias
for var in full_data.columns:
    var_results = stationary[stationary['variable'] == var]
    if var_results.empty:
        print(f"\n{'='*60}")
        print(f'  {var}: NENHUMA transformação estacionária')
        print(f"{'='*60}")
        continue

    print(f"\n{'='*60}")
    print(f'  {var} — {len(var_results)} opções estacionárias')
    print(f"  Recomendado: [{var_results.iloc[0]['pipeline_id']}] {var_results.iloc[0]['pipeline']}")
    print(f"{'='*60}")
    for _, row in var_results.iterrows():
        marker = ' ★' if row.name == var_results.index[0] else '  '
        print(f"{marker} [{int(row['pipeline_id']):>2d}] {row['pipeline']:<25s}  votos={int(row['n_stationary'])}/4  obs={int(row['n_obs'])}")


  ind_extrativa — 10 opções estacionárias
  Recomendado: [1] diff
 ★ [ 1] diff                       votos=4/4  obs=291
   [ 8] sdiff12                    votos=4/4  obs=280
   [13] mom_pct                    votos=4/4  obs=291
   [ 7] yeojohnson→diff            votos=4/4  obs=291
   [ 5] boxcox→diff                votos=4/4  obs=291
   [ 9] sdiff12→diff               votos=4/4  obs=279
   [ 3] log→diff                   votos=4/4  obs=291
   [10] log→sdiff12                votos=4/4  obs=280
   [12] boxcox→sdiff12→diff        votos=4/4  obs=279
   [11] log→sdiff12→diff           votos=4/4  obs=279

  ind_transformacao — 10 opções estacionárias
  Recomendado: [1] diff
 ★ [ 1] diff                       votos=4/4  obs=291
   [ 8] sdiff12                    votos=4/4  obs=280
   [13] mom_pct                    votos=4/4  obs=291
   [ 7] yeojohnson→diff            votos=4/4  obs=291
   [ 5] boxcox→diff                votos=4/4  obs=291
   [ 9] sdiff12→diff               votos=4/4  obs=27

## Exportação CSV

Salva os resultados completos e a recomendação por variável em arquivos CSV.

In [11]:
# %% Exportar resultados
csv_dir = DATA_DIR / 'stationarity'
csv_dir.mkdir(parents=True, exist_ok=True)

# Resultados completos
results_df.to_csv(csv_dir / 'stationarity_results.csv', index=False, sep=';')
print(f'✓ Resultados completos salvos em {csv_dir / "stationarity_results.csv"}')

# Recomendação por variável
best_export = stationary.groupby('variable').first().reset_index()
best_export = best_export[['variable', 'frequency', 'pipeline_id', 'pipeline', 'n_steps', 'n_obs', 'n_stationary']]
best_export.to_csv(csv_dir / 'stationarity_best.csv', index=False, sep=';')
print(f'✓ Recomendações salvas em {csv_dir / "stationarity_best.csv"}')

✓ Resultados completos salvos em C:\Users\lisbo\OneDrive\Documentos\PIB-Nowcast\pib_nowcast\data\stationarity\stationarity_results.csv
✓ Recomendações salvas em C:\Users\lisbo\OneDrive\Documentos\PIB-Nowcast\pib_nowcast\data\stationarity\stationarity_best.csv


In [12]:
end = dt.datetime.now()

diff = end - start

print(f"Tempo de execução: {diff.seconds // 60} minuto(s) e {diff.seconds % 60} segundos")

Tempo de execução: 0 minuto(s) e 48 segundos
